# Video Car Counter Pipeline
## Using Trained SVM Classifier with HOG Features

This notebook implements a complete car-counting pipeline for video analysis:
- **Background Subtraction** for motion detection
- **Contour Detection** to identify moving objects
- **HOG + SVM Classifier** to classify objects as CAR vs NON-CAR
- **Object Tracking** to count unique vehicles crossing a line

Uses the trained model from `Car_Classifier_SVM.ipynb`

In [11]:
# =============================================================================
# IMPORT LIBRARIES
# =============================================================================
import cv2
import numpy as np
import joblib
import os
import math
from skimage.feature import hog
from IPython.display import Video, display, HTML
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [12]:
# =============================================================================
# LOAD TRAINED MODEL AND CONFIGURATION
# =============================================================================

# Load the trained SVM classifier
MODEL_PATH = 'car_classifier_svm.joblib'
CONFIG_PATH = 'car_classifier_config.joblib'

print("Loading trained model...")
svm_classifier = joblib.load(MODEL_PATH)
model_config = joblib.load(CONFIG_PATH)

# Extract configuration
CROP_SIZE = model_config['crop_size']
HOG_ORIENTATIONS = model_config['hog_orientations']
HOG_PIXELS_PER_CELL = model_config['hog_pixels_per_cell']
HOG_CELLS_PER_BLOCK = model_config['hog_cells_per_block']

print("=" * 60)
print("MODEL LOADED SUCCESSFULLY")
print("=" * 60)
print(f"Model accuracy: {model_config['accuracy']*100:.2f}%")
print(f"Crop size: {CROP_SIZE}")
print(f"HOG orientations: {HOG_ORIENTATIONS}")
print(f"HOG pixels per cell: {HOG_PIXELS_PER_CELL}")
print(f"HOG cells per block: {HOG_CELLS_PER_BLOCK}")
print("=" * 60)

Loading trained model...
MODEL LOADED SUCCESSFULLY
Model accuracy: 89.46%
Crop size: (128, 128)
HOG orientations: 9
HOG pixels per cell: (8, 8)
HOG cells per block: (2, 2)


In [13]:
# =============================================================================
# HOG FEATURE EXTRACTION FUNCTION
# =============================================================================

def extract_hog_features(image):
    """
    Extract HOG features from an image region.
    Must match the training configuration exactly.
    """
    # Resize to expected dimensions
    resized = cv2.resize(image, CROP_SIZE)
    
    # Convert to grayscale
    if len(resized.shape) == 3:
        gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    else:
        gray = resized
    
    # Extract HOG features with same parameters as training
    features = hog(
        gray,
        orientations=HOG_ORIENTATIONS,
        pixels_per_cell=HOG_PIXELS_PER_CELL,
        cells_per_block=HOG_CELLS_PER_BLOCK,
        block_norm='L2-Hys',
        visualize=False,
        feature_vector=True
    )
    
    return features


def classify_object(image):
    """
    Classify an image crop as car or non-car.
    
    Returns:
        Tuple: (is_car, confidence)
    """
    features = extract_hog_features(image)
    features = features.reshape(1, -1)
    
    prediction = svm_classifier.predict(features)[0]
    
    if hasattr(svm_classifier, 'predict_proba'):
        proba = svm_classifier.predict_proba(features)[0]
        confidence = proba[prediction]
    else:
        confidence = 1.0
    
    is_car = (prediction == 1)
    return is_car, confidence


print("Feature extraction and classification functions ready!")

Feature extraction and classification functions ready!


In [14]:
# =============================================================================
# OBJECT TRACKER CLASS
# =============================================================================

class EuclideanDistTracker:
    """
    Simple object tracker using Euclidean distance.
    Tracks objects across frames by matching center points.
    """
    def __init__(self, max_distance=50):
        self.center_points = {}  # Store center points of tracked objects
        self.id_count = 0        # Unique ID counter
        self.max_distance = max_distance
    
    def update(self, objects_rect):
        """
        Update tracked objects with new detections.
        
        Args:
            objects_rect: List of [x, y, w, h] bounding boxes
            
        Returns:
            List of [x, y, w, h, id] with assigned IDs
        """
        objects_bbs_ids = []
        
        for rect in objects_rect:
            x, y, w, h = rect
            cx = x + w // 2
            cy = y + h // 2
            
            # Check if this object was detected before
            same_object_detected = False
            for obj_id, pt in self.center_points.items():
                dist = math.hypot(cx - pt[0], cy - pt[1])
                if dist < self.max_distance:
                    self.center_points[obj_id] = (cx, cy)
                    objects_bbs_ids.append([x, y, w, h, obj_id])
                    same_object_detected = True
                    break
            
            # New object detected
            if not same_object_detected:
                self.center_points[self.id_count] = (cx, cy)
                objects_bbs_ids.append([x, y, w, h, self.id_count])
                self.id_count += 1
        
        # Clean up old center points
        new_center_points = {}
        for obj in objects_bbs_ids:
            obj_id = obj[4]
            new_center_points[obj_id] = self.center_points[obj_id]
        self.center_points = new_center_points.copy()
        
        return objects_bbs_ids


print("Object tracker class ready!")

Object tracker class ready!


In [15]:
# =============================================================================
# VIDEO CAR COUNTING PIPELINE
# =============================================================================

def count_cars_in_video(video_path, output_path=None, line_position=0.6, 
                        min_area=1000, max_area=50000, confidence_threshold=0.80):
    """
    Count cars in a video using trained SVM classifier.
    
    Args:
        video_path: Path to input video
        output_path: Path to save output video (optional)
        line_position: Vertical position of counting line (0-1)
        min_area: Minimum contour area to consider
        max_area: Maximum contour area to consider
        confidence_threshold: Minimum confidence for car classification
        
    Returns:
        Dictionary with counting results
    """
    # Check if video exists
    if not os.path.exists(video_path):
        print(f"Error: Video file not found: {video_path}")
        return None
    
    # Open video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video: {video_path}")
        return None
    
    # Get video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print("=" * 60)
    print("VIDEO CAR COUNTING PIPELINE")
    print("=" * 60)
    print(f"Input: {video_path}")
    print(f"Resolution: {width}x{height}")
    print(f"FPS: {fps}")
    print(f"Total frames: {total_frames}")
    print("=" * 60)
    
    # Setup output video writer
    if output_path:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    else:
        out = None
    
    # Initialize components
    tracker = EuclideanDistTracker(max_distance=70)
    bg_subtractor = cv2.createBackgroundSubtractorMOG2(
        history=100, varThreshold=50, detectShadows=True
    )
    
    # Counting line position
    line_y = int(height * line_position)
    offset = 10  # Tolerance for line crossing
    
    # Counting variables
    car_count = 0
    counted_ids = set()
    frame_count = 0
    
    # Statistics
    total_detections = 0
    car_detections = 0
    non_car_detections = 0
    
    print("\nProcessing video...")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_count += 1
        
        # Progress update
        if frame_count % 100 == 0:
            print(f"  Processing frame {frame_count}/{total_frames} ({frame_count/total_frames*100:.1f}%)")
        
        # --- DETECTION PIPELINE ---
        
        # 1. Apply background subtraction
        mask = bg_subtractor.apply(frame)
        
        # 2. Remove shadows (shadow pixels are gray, not white)
        _, mask = cv2.threshold(mask, 254, 255, cv2.THRESH_BINARY)
        
        # 3. Morphological operations to clean up mask
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        mask = cv2.dilate(mask, kernel, iterations=2)
        
        # 4. Find contours
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        # 5. Filter and classify detections
        detections = []
        
        for contour in contours:
            area = cv2.contourArea(contour)
            
            # Filter by area
            if area < min_area or area > max_area:
                continue
            
            # Get bounding box
            x, y, w, h = cv2.boundingRect(contour)
            
            # Filter by aspect ratio (cars are typically wider than tall or roughly square)
            aspect_ratio = w / h if h > 0 else 0
            if aspect_ratio < 0.3 or aspect_ratio > 4.0:
                continue
            
            # Crop the region for classification
            crop = frame[y:y+h, x:x+w]
            if crop.size == 0:
                continue
            
            # Classify using SVM
            is_car, confidence = classify_object(crop)
            
            total_detections += 1
            
            if is_car and confidence >= confidence_threshold:
                car_detections += 1
                detections.append([x, y, w, h])
                
                # Draw green box for cars
                cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
                cv2.putText(frame, f"CAR {confidence:.2f}", (x, y-10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            else:
                non_car_detections += 1
                # Draw red box for non-cars (optional)
                cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 0, 255), 1)
        
        # 6. Update tracker
        tracked_objects = tracker.update(detections)
        
        # 7. Count cars crossing the line
        for obj in tracked_objects:
            x, y, w, h, obj_id = obj
            cy = y + h // 2
            
            # Check if object crosses the counting line
            if line_y - offset < cy < line_y + offset:
                if obj_id not in counted_ids:
                    car_count += 1
                    counted_ids.add(obj_id)
                    print(f"Car #{car_count} counted! (ID: {obj_id})")
        
        # 8. Draw counting line and counter
        cv2.line(frame, (0, line_y), (width, line_y), (0, 255, 255), 3)
        cv2.putText(frame, f"CARS: {car_count}", (20, 60),
                   cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 4)
        cv2.putText(frame, f"Frame: {frame_count}", (20, height - 20),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        # Write output frame
        if out:
            out.write(frame)
    
    # Cleanup
    cap.release()
    if out:
        out.release()
    
    # Results
    results = {
        'total_frames': frame_count,
        'car_count': car_count,
        'total_detections': total_detections,
        'car_detections': car_detections,
        'non_car_detections': non_car_detections,
        'output_video': output_path
    }
    
    print("\n" + "=" * 60)
    print("COUNTING COMPLETE!")
    print("=" * 60)
    print(f"Total cars counted: {car_count}")
    print(f"Total detections: {total_detections}")
    print(f"   - Classified as car: {car_detections}")
    print(f"   - Classified as non-car: {non_car_detections}")
    if output_path:
        print(f"Output video saved to: {output_path}")
    print("=" * 60)
    
    return results


print("Car counting pipeline function ready!")

Car counting pipeline function ready!


## Run Car Counting Pipeline

Specify the input video and run the counting pipeline.

In [16]:
# =============================================================================
# CONFIGURATION - SPECIFY YOUR VIDEO FILE
# =============================================================================

# Input video path - Change this to your video file
# Available videos in workspace:
#   - Vodra/North.mp4
#   - Vodra/South.mp4  
#   - Vodra/weast.mp4
#   - Traffic Dataset/images/test/Video1.mp4 through Video18.mp4

INPUT_VIDEO = "videos/1.mp4"  
OUTPUT_VIDEO = "output/car_count_output.mp4"

# Create output directory if needed
os.makedirs("output", exist_ok=True)

# Check if video exists
if os.path.exists(INPUT_VIDEO):
    print(f"Video file found: {INPUT_VIDEO}")
else:
    print(f"Video file not found: {INPUT_VIDEO}")
    print("\nAvailable videos:")
    for folder in [".", "Vodra", "Traffic Dataset/images/test"]:
        if os.path.exists(folder):
            videos = [f for f in os.listdir(folder) if f.endswith('.mp4')]
            if videos:
                print(f"  {folder}/: {', '.join(videos[:5])}{'...' if len(videos) > 5 else ''}")

Video file found: videos/1.mp4


In [17]:
# =============================================================================
# RUN THE CAR COUNTING PIPELINE
# =============================================================================

# Run the pipeline
results = count_cars_in_video(
    video_path=INPUT_VIDEO,
    output_path=OUTPUT_VIDEO,
    line_position=0.7,      # Counting line at 70% height
    min_area=500,           # Minimum detection area
    max_area=100000,        # Maximum detection area
    confidence_threshold=0.90 # Minimum confidence for car classification (90%+)
)

VIDEO CAR COUNTING PIPELINE
Input: videos/1.mp4
Resolution: 1920x1080
FPS: 25
Total frames: 1501

Processing video...


KeyboardInterrupt: 

In [ ]:
# =============================================================================
# DISPLAY RESULTS SUMMARY
# =============================================================================

if results:
    print("\n" + "=" * 60)
    print("FINAL RESULTS SUMMARY")
    print("=" * 60)
    print(f"""
    ┌────────────────────────────────────────┐
    │  TOTAL CARS COUNTED: {results['car_count']:4d}          │
    ├────────────────────────────────────────┤
    │  Total frames processed: {results['total_frames']:5d}       │
    │  Total object detections: {results['total_detections']:5d}     │
    │  Classified as CAR: {results['car_detections']:5d}           │
    │  Classified as NON-CAR: {results['non_car_detections']:5d}      │
    └────────────────────────────────────────┘
    """)
    
    if results['output_video']:
        print(f"Output video: {results['output_video']}")
        print("\nYou can view the output video with the detections and count overlay.")
else:
    print("Pipeline did not complete successfully.")


FINAL RESULTS SUMMARY

    ┌────────────────────────────────────────┐
    │  TOTAL CARS COUNTED:   36          │
    ├────────────────────────────────────────┤
    │  Total frames processed:  1501       │
    │  Total object detections: 16353     │
    │  Classified as CAR:  3984           │
    │  Classified as NON-CAR: 12369      │
    └────────────────────────────────────────┘
    
Output video: output/car_count_output.mp4

You can view the output video with the detections and count overlay.


## Batch Processing - Run Pipeline on All Videos

Process all videos from the `videos/` folder (1.mp4 through 7.mp4)

In [18]:
# =============================================================================
# BATCH PROCESS ALL 4 VIDEOS
# =============================================================================

# Video folder path
VIDEOS_FOLDER = "videos"

# Store results for all videos
all_results = {}

print("=" * 70)
print("BATCH CAR COUNTING - PROCESSING ALL 4 VIDEOS")
print("=" * 70)

# Process videos 1.mp4 through 4.mp4
for i in range(1, 5):
    video_name = f"{i}.mp4"
    input_path = os.path.join(VIDEOS_FOLDER, video_name)
    output_path = f"output/car_count_{i}.mp4"
    
    print(f"\n{'='*70}")
    print(f"PROCESSING VIDEO {i}/7: {video_name}")
    print(f"{'='*70}")
    
    if os.path.exists(input_path):
        result = count_cars_in_video(
            video_path=input_path,
            output_path=output_path,
            line_position=0.6,
            min_area=500,
            max_area=100000,
            confidence_threshold=0.95
        )
        all_results[video_name] = result
    else:
        print(f"Video not found: {input_path}")
        all_results[video_name] = None

print("\n" + "=" * 70)
print("BATCH PROCESSING COMPLETE!")
print("=" * 70)

BATCH CAR COUNTING - PROCESSING ALL 4 VIDEOS

PROCESSING VIDEO 1/7: 1.mp4
VIDEO CAR COUNTING PIPELINE
Input: videos\1.mp4
Resolution: 1920x1080
FPS: 25
Total frames: 1501

Processing video...
Car #1 counted! (ID: 17)
Car #2 counted! (ID: 92)
Car #3 counted! (ID: 71)
Car #4 counted! (ID: 107)
Car #5 counted! (ID: 108)
  Processing frame 100/1501 (6.7%)
Car #6 counted! (ID: 185)
  Processing frame 200/1501 (13.3%)
Car #7 counted! (ID: 228)
  Processing frame 300/1501 (20.0%)
Car #8 counted! (ID: 289)
  Processing frame 400/1501 (26.6%)
  Processing frame 500/1501 (33.3%)
Car #9 counted! (ID: 380)
  Processing frame 600/1501 (40.0%)
Car #10 counted! (ID: 411)
  Processing frame 700/1501 (46.6%)
  Processing frame 800/1501 (53.3%)
Car #11 counted! (ID: 491)
Car #12 counted! (ID: 533)
Car #13 counted! (ID: 497)
  Processing frame 900/1501 (60.0%)
Car #14 counted! (ID: 624)
Car #15 counted! (ID: 636)
  Processing frame 1000/1501 (66.6%)
Car #16 counted! (ID: 769)
Car #17 counted! (ID: 770)
C

KeyboardInterrupt: 

In [ ]:
# =============================================================================
# SUMMARY TABLE FOR ALL VIDEOS
# =============================================================================

print("\n" + "=" * 70)
print("FINAL SUMMARY - ALL VIDEOS")
print("=" * 70)
print(f"\n{'Video':<12} {'Cars':<8} {'Frames':<10} {'Detections':<12} {'Output':<25}")
print("-" * 70)

total_cars = 0
total_frames = 0

for video_name, result in all_results.items():
    if result:
        total_cars += result['car_count']
        total_frames += result['total_frames']
        output_file = result['output_video'].split('/')[-1] if result['output_video'] else 'N/A'
        print(f"{video_name:<12} {result['car_count']:<8} {result['total_frames']:<10} {result['total_detections']:<12} {output_file:<25}")
    else:
        print(f"{video_name:<12} {'ERROR':<8} {'-':<10} {'-':<12} {'N/A':<25}")

print("-" * 70)
print(f"{'TOTAL':<12} {total_cars:<8} {total_frames:<10}")
print("=" * 70)

print(f"\nGRAND TOTAL: {total_cars} cars counted across all videos!")
print(f"Output videos saved in: output/car_count_1.mp4 through car_count_7.mp4")